# Taller 1: Introducción a Python para modelación hidrológica

**ICYA 4710 · Modelación de Sistemas y Procesos Hidrológicos**
Segundo semestre de 2026 · Juan Sebastián Hernández S., Ph.D.

---

## De qué se trata este taller

En el tutorial de configuración usted configuró el entorno WSL, conda, VS Code, Git y SPOTPY.
Hoy lo vamos a usar. La idea no es aprender Python en abstracto, sino aprender **el Python que se
necesita para recorrer el protocolo de modelación** que vimos en clase.

Cada parte del taller corresponde a una caja del protocolo:

| Parte del taller | Caja del protocolo | Minutos |
|---|---|---|
| 0. Verificación del entorno | — | 10 |
| 1. Python en 30 minutos | — | 30 |
| 2. Obtener y evaluar los datos | *Obtener datos medidos* · *Evaluar datos y necesidades de datos* | 20 |
| 3. HYMOD: del modelo conceptual al código | *Conceptualizar el modelo* · *Seleccionar o desarrollar el código* | 15 |
| 4. Criterios de desempeño | *Determinar criterios de desempeño aceptable* | 18 |
| 5. Calibración por ensayo y error | *Estimación de parámetros* | 12 |
| 6. Dos cuencas colombianas | *vuelta al inicio: propósito y datos* | 15 |

**Todo el código de este cuaderno ya está escrito y funciona.** Su trabajo no es programar, sino
**decidir**, **ejecutar**, **mirar el resultado** e **interpretarlo**. Las celdas marcadas con ✏️ son las que usted debe completar o responder.

---

## Qué se entrega

1. Este cuaderno completo, **corrido de arriba a abajo sin errores**, con las celdas ✏️ resueltas
   (6 de código y 10 de respuesta escrita).
2. Publicado en un repositorio propio de GitHub (las instrucciones están al final).

**Fecha límite:** una semana después de la sesión. La rúbrica está en el enunciado del taller.

---

## Regla de oro del taller

> Un modelo que reproduce bien los caudales observados **no es necesariamente un buen modelo**.
> Puede estar dando las respuestas correctas con los argumentos equivocados. Buena parte de lo que
> haremos hoy consiste en aprender a desconfiar de un hidrograma que aparenta ser correcto.

---
# Parte 0 · Verificación del *environment*

Antes de empezar, confirme que el entorno que armó en el tutorial está activo.

**Antes de correr la celda siguiente**, revise la esquina superior derecha de VS Code: el kernel debe
decir `hidro` (o el nombre del environment que usted creó). Si dice `base` o `Python 3.x`, cámbielo con
`Ctrl + Shift + P` → `Notebook: Select Notebook Kernel`.

In [ ]:
# --- Celda de verificacion: si esto corre, el entorno esta bien ---
import sys, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python     :", sys.version.split()[0])
print("Ejecutable :", sys.executable)
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("Sistema    :", platform.system(), platform.release())

# Los datos se encuentran en la carpeta 'datos/' del repositorio, un nivel arriba de 'notebooks/'
DATOS = Path("..") / "datos"
if not DATOS.exists():
    DATOS = Path("datos")          # por si abre el cuaderno desde la raiz del repositorio

archivos = sorted(p.name for p in DATOS.glob("*.csv"))
print("\nCarpeta de datos:", DATOS.resolve())
print("Archivos encontrados:", archivos)

assert "hymod_input.csv" in archivos, "No encuentro hymod_input.csv. Revise la ruta DATOS."
print("\nTodo listo.")

**Si el `Ejecutable` no contiene `envs/hidro`**, el kernel está apuntando al Python equivocado.
Vuelva a la sección 4.4 del tutorial. Si `assert` falla, revise que clonó el repositorio completo
del taller y que está abriendo el cuaderno desde `notebooks/`.

Una última línea de configuración, para que todas las gráficas del taller sean legibles:

In [ ]:
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi": 600,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

---
# Parte 1: Python en 30 minutos

Todo lo que sigue lo va a necesitar en las Partes 2 a 6. No hay nada aquí que no reaparezca después.

## 1.1 Variables y tipos

Una variable es un nombre que apunta a un valor. Python deduce el tipo por sí mismo.

In [ ]:
area = 446.82            # km2          -> float (numero decimal)
n_años = 8               # años         -> int   (numero entero)
estacion = "21167080"    # codigo IDEAM -> str   (texto)
calibrada = False        #              -> bool  (verdadero / falso)

for v in (area, n_años, estacion, calibrada):
    print(f"{str(v):>10}  ->  {type(v).__name__}")

**Nota:** el nombre de la variable es para las personas, no para el computador. `area` se entiende;
`a1` no. En un script de modelación con veinte parámetros, esa diferencia puede hacer que su código se ininteligible.

## 1.2 Texto y f-strings

Las *f-strings* arman texto mezclando variables. Son la forma más limpia de imprimir resultados.

In [ ]:
caudal_medio = 19.9054321      # m3/s

print(f"Estacion {estacion}: area = {area} km2")
print(f"Caudal medio = {caudal_medio} m3/s")          # ilegible
print(f"Caudal medio = {caudal_medio:.2f} m3/s")      # 2 decimales
print(f"Caudal medio = {caudal_medio:8.2f} m3/s")     # 2 decimales, ancho 8
print(f"Rendimiento  = {caudal_medio/area*1000:.2f} L/s/km2")

## 1.3 Listas, y por qué vamos a usar `numpy`

Una **lista** guarda cosas en orden. Sirve para todo, pero no sirve para hacer operaciones aritméticas o cálculos más avanzados de manera sencilla o directa.

In [ ]:
# Precipitacion diaria de una semana (mm)
p = [0.0, 12.4, 3.1, 0.0, 0.0, 28.7, 5.2]

print("elementos      :", len(p))
print("primer dia     :", p[0])        # Python empieza a contar en 0
print("ultimo dia     :", p[-1])       # los indices negativos cuentan desde el final
print("dias 2 a 4     :", p[1:4])      # el limite superior NO se incluye
print("lista + lista  :", p + [1.0])   # el signo + CONCATENA, no suma

In [ ]:
# Con numpy, el mismo signo + hace aritmética elemento a elemento
P = np.array(p)

print("P            :", P)
print("P * 2        :", P * 2)
print("P acumulada  :", P.cumsum())
print(f"total  = {P.sum():.1f} mm")
print(f"media  = {P.mean():.2f} mm/dia")
print(f"maximo = {P.max():.1f} mm  (dia {P.argmax() + 1})")

**Regla práctica:** listas para colecciones de cosas heterogéneas (nombres de archivos, rutas,
etiquetas); `numpy` para series numéricas. Una serie de 30 años de caudal diario son 10 957 números.
`numpy` la procesa en milisegundos, una lista de Python no.

Un truco que va a usar todo el tiempo: **máscaras booleanas**.

In [ ]:
mascara = P > 0                      # array de True/False, del mismo tamano
print("máscara       :", mascara)
print("días con lluvia:", mascara.sum())          # True cuenta como 1
print("valores        :", P[mascara])             # filtra el array
print(f"lluvia media en días húmedos: {P[mascara].mean():.2f} mm")

## 1.4 Control de flujo: `for` e `if`

Un `for` repite mientras que un `if` decide. Usarlos conjuntamente es casi toda la programación que necesita un modelo conceptual.

In [ ]:
for i, lluvia in enumerate(P):
    if lluvia == 0:
        clase = "seco"
    elif lluvia < 10:
        clase = "lluvia débil"
    else:
        clase = "lluvia fuerte"
    print(f"día {i}:  {lluvia:5.1f} mm   {clase}")

**Atención a la indentación.** En Python los bloques se definen con espacios, no con llaves.
Las cuatro líneas dentro del `for` están indentadas cuatro espacios. Si desalinea una, el programa
cambia de significado o falla.

## 1.5 Funciones

Una función empaqueta un cálculo con un nombre. A continuación, un par de funciones para convertir caudales en escorrentía, y vice versa:

In [ ]:
def m3s_a_mm(Q_m3s, area_km2):
    """Convierte un caudal [m3/s] a lámina de agua diaria [mm/dia].

    Un caudal de 1 m3/s durante un día son 86400 m3 de agua. Repartidos sobre
    el área de la cuenca (en m2) dan una altura en metros, que pasamos a mm.
    """
    return Q_m3s * 86400.0 / (area_km2 * 1e6) * 1000.0


def mm_a_m3s(Q_mm, area_km2):
    """Operacion inversa de m3s_a_mm."""
    return Q_mm * (area_km2 * 1e6) / 1000.0 / 86400.0


q_mm = m3s_a_mm(caudal_medio, area)
print(f"{caudal_medio:.2f} m3/s en {area} km2  =  {q_mm:.3f} mm/dia")
print(f"y de vuelta: {mm_a_m3s(q_mm, area):.2f} m3/s")

**Por qué importa esta conversión:** los modelos conceptuales de lluvia-escorrentía trabajan en
**láminas de agua** (mm), porque así la precipitación, la evapotranspiración y el caudal son comparables y el
balance hídrico cierra sumando y restando. Los caudales observados vienen en **m³/s**. Convertir mal
las unidades es, con diferencia, el error más frecuente al montar un modelo.

## 1.6 Cuando algo falla, cómo leer el error

En clase van a aparecer errores. Todos. La habilidad que sí importa es leer la última línea del
mensaje. Estos son los cuatro que va a ver hoy:

In [ ]:
ejemplos = [
    ("NameError",         lambda: precipitacion_total),      # la variable no existe (o la escribio mal)
    ("TypeError",         lambda: "3" + 3),                  # mezcló texto con número
    ("IndexError",        lambda: p[99]),                    # pidió la posición 99 de una lista de 7
    ("ZeroDivisionError", lambda: 1 / 0),                    # división por cero
]

for nombre, f in ejemplos:
    try:
        f()
    except Exception as e:
        print(f"{nombre:18} -> {type(e).__name__}: {e}")

Y el quinto, que no se puede demostrar sin romper el notebook:

- **`ModuleNotFoundError: No module named 'xxx'`**: casi nunca es que el paquete no esté instalado.
  Es que está instalado en **otro** environment. Revise el kernel (esquina superior derecha) y
  ejecute `conda env list` en la terminal: el `*` le dice dónde está ubicado.

**Regla:** cuando algo falle, lea la **última** línea del mensaje primero, y después mire el número de
línea que le señala. No borre y reescriba a ciegas.

## 1.7 `pandas`: las tablas/bases de datos con las que vamos a trabajar

`pandas` es `numpy` con etiquetas. Un `DataFrame` es una tabla mientras que un `Series` es una columna. Lo que lo
hace indispensable en hidrología es que el índice puede ser una **fecha**.

In [ ]:
demo = pd.DataFrame({
    "P_mm":  [0.0, 12.4, 3.1, 0.0, 0.0, 28.7, 5.2],
    "Q_m3s": [3.2,  3.4, 6.8, 5.1, 4.2,  4.8, 12.5],
}, index=pd.date_range("2001-04-01", periods=7, freq="D"))
demo.index.name = "fecha"

print(demo)
print("\nSolo la columna de caudal:")
print(demo["Q_m3s"])
print("\nUn dia especifico:")
print(demo.loc["2001-04-03"])

In [ ]:
# Crear columnas nuevas es una línea
demo["Q_mm"] = m3s_a_mm(demo["Q_m3s"], area)
demo["mes"]  = demo.index.month

print(demo.round(3))
print("\nEstadisticos de la tabla:")
print(demo[["P_mm", "Q_mm"]].describe().round(3))

---
## ✏️ Ejercicio 1

**1a.** Complete la función `coef_escorrentia`, que recibe dos series (o arrays) de precipitación y
caudal **en las mismas unidades** y devuelve el coeficiente de escorrentía $C = \sum Q / \sum P$.
Es una sola línea.

In [ ]:
def coef_escorrentia(P, Q):
    """Coeficiente de escorrentia: fraccion de la lluvia que sale como caudal."""
    # ✏️ COMPLETE: devuelva la suma de Q dividida entre la suma de P
    return ...


# Verificación (no modifique): debe imprimir 0.4
prueba = coef_escorrentia(np.array([10.0, 20.0, 20.0]), np.array([4.0, 8.0, 8.0]))
print("C =", prueba)
assert prueba is not Ellipsis, "Todavia no completo la funcion: reemplace el '...' del return."
assert abs(prueba - 0.4) < 1e-9, f"La funcion devuelve {prueba}, pero deberia devolver 0.4."
print("Correcto.")

**1b.** El coeficiente de escorrentía que acaba de programar puede dar un valor **mayor que 1**
en una cuenca real. Escriba dos razones distintas por las que eso puede ocurrir. Piense en el
protocolo: una razón debe ser física (algo que pasa en la cuenca) y la otra debe ser de datos.

> ✏️ **Respuesta 1b:**
> 1. *(escriba aquí)*
> 2. *(escriba aquí)*

---
# Parte 2 · Obtener y evaluar los datos

> **Protocolo:** cajas *Obtener datos medidos de entrada y salida* y *Evaluar datos y necesidades de datos*.

Ninguna decisión de modelación se toma antes de mirar los datos. Esta parte es corta de código y larga
de criterio.

Vamos a trabajar con la cuenca de ejemplo que trae SPOTPY: una cuenca experimental de **1.783 km²**
con cinco años de datos diarios (2012–2016) de precipitación, evapotranspiración potencial (estimada
con la fórmula de Turc) y caudal.

## 2.1 Cargar la serie

In [ ]:
AREA_HYMOD = 1.783        # km2

raw = pd.read_csv(
    DATOS / "hymod_input.csv",
    sep=";",                              # este archivo usa punto y coma
    parse_dates=["Date"],
    date_format="%d.%m.%Y",               # dia.mes.año
)
raw.head()

In [ ]:
# Renombrar a algo manejable y poner la fecha como índice
df = raw.rename(columns={
    "Date": "fecha",
    "rainfall[mm]": "P_mm",
    "TURC [mm d-1]": "ETP_mm",
    "Discharge[ls-1]": "Q_ls",
}).set_index("fecha")

# Unidades: de L/s a m3/s, y de m3/s a lámina de agua (mm/dia)
df["Q_m3s"] = df["Q_ls"] / 1000.0
df["Q_mm"]  = m3s_a_mm(df["Q_m3s"], AREA_HYMOD)
df = df[["P_mm", "ETP_mm", "Q_m3s", "Q_mm"]]

df.head()

## 2.2 Primera inspección

Tres comandos que se ejecutan **siempre**, antes de cualquier otra cosa:

In [ ]:
print(f"Periodo : {df.index.min().date()}  a  {df.index.max().date()}")
print(f"Registros: {len(df)}   (días esperados: {(df.index.max()-df.index.min()).days + 1})")
print()
df.info()

In [ ]:
df.describe().round(3)

In [ ]:
# Faltantes por año y por variable
faltantes = df.isna().groupby(df.index.year).sum()
faltantes["dias"] = df.groupby(df.index.year).size()
faltantes

**Lea esa tabla antes de seguir.** Toda la serie de caudal de 2012 está vacía. Eso no es un
accidente, ya que en esta cuenca el aforo empezó en 2013. Es decir, el primer año **solo** sirve como período
de calentamiento (que es exactamente lo que necesitamos).

## 2.3 Balance hídrico anual

La primera prueba de credibilidad de un conjunto de datos no es estadística, es el balance hídrico.
Sobre un año hidrológico completo, y con almacenamiento aproximadamente estable:

$$P \approx ET_{real} + Q$$

de donde $ET_{real} \approx P - Q$. Y siempre debe cumplirse que $ET_{real} \le ETP$.

In [ ]:
anual = df.resample("YS").sum(min_count=300)          # min_count: no sumar años incompletos
anual = anual[["P_mm", "ETP_mm", "Q_mm"]]
anual["ET_implicita_mm"] = anual["P_mm"] - anual["Q_mm"]
anual["C_escorrentia"]   = anual["Q_mm"] / anual["P_mm"]
anual["ET_impl/ETP"]     = anual["ET_implicita_mm"] / anual["ETP_mm"]
anual.index = anual.index.year
anual.round(2)

**Cómo se lee esta tabla:**

- `C_escorrentia` es la fracción de la lluvia que sale por el río. Valores típicos: 0.2–0.4 en cuencas
  secas, 0.5–0.8 en cuencas húmedas de montaña.
- `ET_impl/ETP` es la relación entre la evapotranspiración que exige el balance hídrico y la máxima
  que permite la atmósfera. **Si es mayor que 1, los datos son internamente inconsistentes**: alguna
  de las tres series está mal (o la cuenca importa o exporta agua por vías que no estamos midiendo).
  Es la revisión de la diapositiva "verificar que el modelo no esté dando las respuestas correctas con
  argumentos equivocados", pero aplicada a los datos, antes de tener modelo.

## 2.4 La gráfica de siempre: hietograma e hidrograma

En hidrología, esta figura se dibuja con la lluvia colgando invertida desde arriba. Así se ve casi de inmediato qué evento produjo qué creciente.

In [ ]:
def hietograma_hidrograma(datos, titulo="", ax=None):
    """Gráfica caudal (abajo) y precipitacion invertida (arriba) en el mismo eje temporal."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4.5))

    ax.plot(datos.index, datos["Q_mm"], color="#1f4e79", lw=1.0, label="Q observado")
    ax.set_ylabel("Caudal (mm/dia)")
    ax.set_ylim(0, datos["Q_mm"].max() * 2.2)
    ax.set_xlabel("Fecha")

    axp = ax.twinx()
    axp.bar(datos.index, datos["P_mm"], width=1.0, color="#5b9bd5", alpha=0.7, label="Precipitación")
    axp.set_ylabel("Precipitacion (mm/dia)")
    axp.invert_yaxis()
    axp.set_ylim(datos["P_mm"].max() * 2.6, 0)
    axp.grid(False)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axp.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="center right", framealpha=0.9)
    ax.set_title(titulo)
    return ax


hietograma_hidrograma(df, "Cuenca de ejemplo (1.783 km2) serie completa 2012-2016")
plt.tight_layout(); plt.show()

In [ ]:
# Un acercamiento a un solo año deja ver la respuesta a eventos individuales
hietograma_hidrograma(df.loc["2014"], "Acercamiento: 2014")
plt.tight_layout(); plt.show()

## 2.5 Curva de duración de caudales

La curva de duración responde a la siguiente pregunta: *¿qué porcentaje del tiempo es excedido un caudal dado?* Es una de las formas más compactas de describir el régimen de una cuenca y va a reaparecer en la Parte 4, cuando discutamos qué favorece cada función objetivo.

In [ ]:
def curva_duracion(Q):
    """Devuelve (probabilidad de excedencia %, caudal ordenado de mayor a menor)."""
    q = np.sort(np.asarray(Q)[np.isfinite(Q)])[::-1]
    prob = np.arange(1, len(q) + 1) / (len(q) + 1) * 100
    return prob, q


prob, q = curva_duracion(df["Q_mm"])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(prob, q, color="#1f4e79", lw=1.8)
ax.set_yscale("log")
ax.set_xlabel("Probabilidad de excedencia (%)")
ax.set_ylabel("Caudal (mm/dia)")
ax.set_title("Curva de duracion de caudales")
for p_ in (5, 50, 95):
    v = np.interp(p_, prob, q)
    ax.axvline(p_, color="grey", ls=":", lw=1)
    ax.annotate(f"Q{p_} = {v:.2f}", xy=(p_, v), xytext=(p_ + 3, v * 1.6), fontsize=9)
plt.tight_layout(); plt.show()

---
## ✏️ Ejercicio 2: Particionar la serie

Antes de calibrar un modelo, hay que decidir **qué parte de la serie se usa para qué**. El protocolo separa
explícitamente el *conjunto para estimación de parámetros* del *conjunto para prueba de
aceptabilidad*, y además necesitamos un período de **calentamiento** para que la condición inicial
supuesta (todos los almacenamientos en cero) no influya sobre los resultados.

Complete las tres fechas abajo. Criterios que debe respetar:

- El calentamiento debe cubrir, como mínimo, el período **sin datos de caudal**.
- Los dos períodos restantes deben ser **disjuntos** y de longitud razonable.
- Los tres juntos deben cubrir toda la serie disponible.

**Justifique su partición en la celda de texto que sigue.** No hay una única respuesta correcta, pero
sí hay respuestas indefendibles.

In [ ]:
# ✏️ COMPLETE las tres fechas (formato "AAAA-MM-DD")

FIN_CALENTAMIENTO   = "...."     # ultimo dia que se descarta
INICIO_CALIBRACION  = "...."     # primer dia del conjunto de estimacion de parametros
INICIO_ACEPTABILIDAD = "...."    # primer dia del conjunto de prueba de aceptabilidad

# --- No modifique de aqui en adelante ---
for _nombre, _valor in [("FIN_CALENTAMIENTO", FIN_CALENTAMIENTO),
                        ("INICIO_CALIBRACION", INICIO_CALIBRACION),
                        ("INICIO_ACEPTABILIDAD", INICIO_ACEPTABILIDAD)]:
    assert _valor != "....", (
        f"Falta completar {_nombre}. Escriba una fecha en formato \"AAAA-MM-DD\" "
        "en las tres variables de arriba antes de correr esta celda.")

CAL = slice(INICIO_CALIBRACION, pd.Timestamp(INICIO_ACEPTABILIDAD) - pd.Timedelta(days=1))
ACE = slice(INICIO_ACEPTABILIDAD, str(df.index.max().date()))

print(f"Calentamiento : {df.index.min().date()} -> {FIN_CALENTAMIENTO}   ({len(df.loc[:FIN_CALENTAMIENTO])} dias)")
print(f"Calibracion   : {CAL.start} -> {CAL.stop.date()}   ({len(df.loc[CAL])} dias, "
      f"{df.loc[CAL,'Q_mm'].notna().sum()} con caudal observado)")
print(f"Aceptabilidad : {ACE.start} -> {ACE.stop}   ({len(df.loc[ACE])} dias, "
      f"{df.loc[ACE,'Q_mm'].notna().sum()} con caudal observado)")

> ✏️ **Justificación (3–5 líneas):**
> *(escriba aquí por qué eligió esas fechas: cuánto calentamiento, cómo repartió el resto y qué
> tuvo en cuenta sobre los años húmedos y secos que vio en la gráfica)*

---
# Parte 3: HYMOD, del modelo conceptual al código

> **Protocolo:** cajas *Conceptualizar el modelo* y *Seleccionar o desarrollar el código*.

## 3.1 El modelo conceptual

HYMOD es un modelo **agregado** (toda la cuenca es una caja (o punto) que recibe unas entradas y devuelve una salida), **conceptual** (los almacenamientos no corresponden a volúmenes medibles) y de **paso diario**. Tiene dos bloques:

**Bloque 1, Producción de escorrentía.** La cuenca no se satura de golpe debido a que unas regiones o zonas se saturan
antes que otras. HYMOD representa eso con una *distribución de capacidades de almacenamiento*, en donde la
fracción de la cuenca con capacidad menor o igual a $c$ es

$$F(c) = 1 - \left(1 - \frac{c}{c_{max}}\right)^{b}, \qquad 0 \le c \le c_{max}$$

Con $b$ pequeño, la cuenca es casi homogénea. Con $b$ grande, hay mucha heterogeneidad y se genera
escorrentía desde el comienzo del evento. El almacenamiento máximo de la cuenca es
$S_{max} = c_{max}/(b+1)$.

La evapotranspiración real se limita según la humedad disponible.

$$ET_{real} = \frac{S}{S_{max}}\, ETP \qquad \text{y siempre } ET_{real} \le ETP$$

**Bloque 2, Tránsito.** El exceso de precipitación se reparte. Una fracción $\alpha$ va a una cascada
de **tres embalses lineales rápidos** (respuesta de crecientes) y la fracción $1-\alpha$ va a **un
embalse lineal lento** (flujo base). Un embalse lineal es simplemente

$$S_{t} = (1-K)\,(S_{t-1} + I_t), \qquad Q_t = \frac{K}{1-K}\,S_t$$

## 3.2 Los cinco parámetros

| Parámetro | Significado físico aproximado | Rango usual | Efecto si aumenta |
|---|---|---|---|
| `cmax`  | Capacidad máxima de almacenamiento de suelo (mm) | 1 – 500 | Más agua se retiene: menos escorrentía total |
| `b`     | Heterogeneidad espacial de esa capacidad (–) | 0.1 – 2.0 | Se genera escorrentía más temprano en el evento |
| `alpha` | Fracción del exceso que va al flujo rápido (–) | 0.1 – 0.99 | Picos más altos, recesiones más cortas |
| `Ks`    | Constante del embalse lento (1/día) | 0.001 – 0.10 | Flujo base se agota más rápido |
| `Kq`    | Constante de los embalses rápidos (1/día) | 0.1 – 0.99 | Picos más agudos y adelantados |

**Nota importante:** estos parámetros son *conceptuales*. `cmax` no es una capacidad de campo que
usted pueda ir a medir directamente. Se estiman calibrando, y por eso la interpretación física de
los valores calibrados es limitada.

In [ ]:
def hymod(P, ETP, cmax, b, alpha, Ks, Kq, devolver_estados=False):
    """Modelo lluvia-escorrentia HYMOD, paso diario, unidades en mm.

    Parámetros
    ----------
    P, ETP : array de precipitación y evapotranspiración potencial (mm/dia)
    cmax   : capacidad máxima de almacenamiento del suelo (mm)
    b      : exponente de la distribución de capacidades (-)
    alpha  : fracción del exceso que va al flujo rápido (-)
    Ks, Kq : constantes de los embalses lento y rápidos (1/dia)

    Devuelve
    --------
    array de caudal simulado (mm/dia), o un diccionario con los estados internos
    si devolver_estados=True.
    """
    P = np.asarray(P, dtype=float)
    ETP = np.asarray(ETP, dtype=float)
    n = len(P)

    Qsim = np.zeros(n); ETr = np.zeros(n); Hs = np.zeros(n)
    Qrap = np.zeros(n); Qlen = np.zeros(n); Exc = np.zeros(n)

    S = 0.0                       # humedad del suelo (mm)  <- condicion inicial
    S_lento = 0.0                 # tanque lento
    S_rapido = [0.0, 0.0, 0.0]    # cascada de tres tanques rapidos
    S_max = cmax / (b + 1.0)      # almacenamiento maximo de la cuenca (mm)

    for t in range(n):
        # 1) Produccion de escorrentia -----------------------------------
        c_ant = cmax * (1.0 - max(1.0 - S / S_max, 0.0) ** (1.0 / (b + 1.0)))
        ER1 = max(P[t] - cmax + c_ant, 0.0)                 # exceso por saturacion total
        Pn = P[t] - ER1
        frac = min((c_ant + Pn) / cmax, 1.0)
        S_nuevo = S_max * (1.0 - (1.0 - frac) ** (b + 1.0))
        ER2 = max(Pn - (S_nuevo - S), 0.0)                  # exceso por saturacion parcial

        # 2) Evapotranspiracion real, limitada por la humedad -------------
        et = min((S_nuevo / S_max) * ETP[t], S_nuevo)
        S = max(S_nuevo - et, 0.0)

        # 3) Reparto rápido / lento --------------------------------------
        exceso = ER1 + ER2
        U_rapido = alpha * exceso
        U_lento = (1.0 - alpha) * exceso

        # 4) Tránsito por embalses lineales ------------------------------
        S_lento = (1.0 - Ks) * (S_lento + U_lento)
        q_lento = (Ks / (1.0 - Ks)) * S_lento

        entrada = U_rapido
        for i in range(3):
            S_rapido[i] = (1.0 - Kq) * (S_rapido[i] + entrada)
            entrada = (Kq / (1.0 - Kq)) * S_rapido[i]
        q_rapido = entrada

        Qsim[t] = q_lento + q_rapido
        ETr[t] = et; Hs[t] = S; Qrap[t] = q_rapido; Qlen[t] = q_lento; Exc[t] = exceso

    if devolver_estados:
        return {"Q": Qsim, "ET": ETr, "humedad_suelo": Hs,
                "Q_rapido": Qrap, "Q_lento": Qlen, "exceso": Exc, "S_max": S_max}
    return Qsim


print("HYMOD listo.")

## 3.3 Primera corrida

Antes de calibrar el modelo, se debe correr con un conjunto de parámetros **a priori** (valores razonables
tomados de la literatura o de cuencas parecidas) y se evalúea la respuesta. Esto es lo que el protocolo llama
*ejercicios preliminares de cuantificación, para comenzar a sentir la respuesta del caso de estudio*.

In [ ]:
PAR_INICIAL = dict(cmax=412.33, b=0.173, alpha=0.813, Ks=0.040, Kq=0.559)

salida = hymod(df["P_mm"].values, df["ETP_mm"].values, **PAR_INICIAL, devolver_estados=True)
df["Q_sim"] = salida["Q"]

print("Parametros a priori:", PAR_INICIAL)
print(f"S_max = {salida['S_max']:.1f} mm")
print()
print(f"Q observado medio  : {df['Q_mm'].mean():.3f} mm/dia")
print(f"Q simulado medio   : {df['Q_sim'].mean():.3f} mm/dia")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
d = df.loc["2013":]
ax.plot(d.index, d["Q_mm"],  color="#c00000", lw=0.9, label="Q observado")
ax.plot(d.index, d["Q_sim"], color="#1f4e79", lw=1.1, label="Q simulado (a priori)")
ax.set_ylabel("Caudal (mm/dia)"); ax.set_xlabel("Fecha")
ax.set_title("Primera corrida de HYMOD, sin calibrar")
ax.legend()
plt.tight_layout(); plt.show()

## 3.4 Mirar por dentro del modelo

Un hidrograma de salida no dice si el modelo está funcionando por las razones correctas. Los estados
internos sí. Estas tres gráficas son la traducción directa de los puntos 5 a 8 de la diapositiva de
credibilidad del protocolo.

In [ ]:
estados = pd.DataFrame({
    "humedad_suelo": salida["humedad_suelo"],
    "ET_real":       salida["ET"],
    "Q_rápido":      salida["Q_rapido"],
    "Q_lento":       salida["Q_lento"],
}, index=df.index)

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axs[0].plot(estados.index, estados["humedad_suelo"], color="#7f6000", lw=1)
axs[0].axhline(salida["S_max"], color="grey", ls="--", lw=1, label="S_max")
axs[0].set_ylabel("Humedad del\nsuelo (mm)"); axs[0].legend(loc="upper right")
axs[0].set_title("Estados internos de HYMOD con los parametros a priori")

axs[1].plot(df.index, df["ETP_mm"], color="#bfbfbf", lw=0.9, label="ETP (potencial)")
axs[1].plot(estados.index, estados["ET_real"], color="#2e7d32", lw=1.0, label="ET real simulada")
axs[1].set_ylabel("ET (mm/dia)"); axs[1].legend(loc="upper right")

axs[2].stackplot(estados.index, estados["Q_lento"], estados["Q_rápido"],
                 labels=["Flujo lento (base)", "Flujo rápido"],
                 colors=["#9dc3e6", "#1f4e79"])
axs[2].set_ylabel("Caudal (mm/dia)"); axs[2].set_xlabel("Fecha")
axs[2].legend(loc="upper right")

plt.tight_layout(); plt.show()

---
## ✏️ Ejercicio 3 — Coherencia interna

La celda siguiente calcula tres verificaciones. **Córrala y luego responda.**

In [ ]:
P_tot  = df["P_mm"].sum()
ET_tot = estados["ET_real"].sum()
Q_tot  = df["Q_sim"].sum()
dS     = estados["humedad_suelo"].iloc[-1] - 0.0     # cambio de almacenamiento del suelo

print("--- Verificación 1: balance de masa del modelo ---")
print(f"P = {P_tot:8.1f} mm     ET = {ET_tot:8.1f} mm     Q = {Q_tot:8.1f} mm")
print(f"Residual P - ET - Q - dS_suelo = {P_tot - ET_tot - Q_tot - dS:8.1f} mm")
print("   (el residual que queda es el agua atrapada en los cuatro embalses de transito)")

print("\n--- Verificación 2: ET real vs ETP ---")
print(f"Dias en que ET_real > ETP: {(estados['ET_real'] > df['ETP_mm'] + 1e-9).sum()}")
print(f"Relacion ET_real / ETP (media): {ET_tot / df['ETP_mm'].sum():.2f}")

print("\n--- Verificación 3: índice de flujo base simulado ---")
print(f"Fraccion del caudal que sale por el embalse lento: "
      f"{estados['Q_lento'].sum() / (Q_tot):.2f}")

> ✏️ **Respuesta 3.**
>
> **3a.** ¿La verificación 2 se cumple? Explique en una línea por qué el modelo *no puede*
> producir `ET_real > ETP`, mirando la línea del código donde se calcula `et`.
>
> *(escriba aquí)*
>
> **3b.** Compare la fracción de flujo base **simulada** (verificación 3) con lo que sugiere el
> hidrograma **observado** de la Parte 2. ¿Le parece que el modelo está repartiendo bien el agua entre
> flujo rápido y flujo base? ¿Qué parámetro tocaría para corregirlo?
>
> *(escriba aquí)*

---
# Parte 4 · Criterios de desempeño aceptable

> **Protocolo:** caja *Determinar criterios de desempeño aceptable*.

Una función objetivo (FO) resume en un número la distancia entre lo simulado y lo observado. El punto
de esta parte es que **no existe la FO correcta**: cada una favorece un aspecto distinto, y elegir cuál
usar es una decisión de modelación tan importante como elegir el modelo.

## 4.1 Las cinco funciones objetivo del taller

$$NSE = 1 - \frac{\sum (Q_{sim} - Q_{obs})^2}{\sum (Q_{obs} - \overline{Q_{obs}})^2}
\qquad\text{(1 es perfecto; 0 equivale a usar la media observada)}$$

$$KGE = 1 - \sqrt{(r-1)^2 + (\beta-1)^2 + (\gamma-1)^2}$$

donde $r$ es la correlación, $\beta = \overline{Q_{sim}}/\overline{Q_{obs}}$ el sesgo de volumen y
$\gamma$ la relación de coeficientes de variación. El KGE es útil justamente porque **separa** esos
tres errores en lugar de mezclarlos.

$$PBIAS = 100\,\frac{\sum (Q_{sim} - Q_{obs})}{\sum Q_{obs}}
\qquad\text{(negativo = el modelo subestima el volumen)}$$

El **logNSE** es el NSE calculado sobre $\log Q$: comprime los picos y amplifica los caudales bajos,
así que mide qué tan bien se reproduce el estiaje. El **RMSE** está en las unidades del caudal, lo que
lo hace fácil de interpretar pero dificulta comparar entre cuencas.

In [ ]:
def _limpiar(obs, sim):
    """Descarta los dias sin observacion. Toda FO debe hacer esto primero."""
    obs = np.asarray(obs, dtype=float); sim = np.asarray(sim, dtype=float)
    m = np.isfinite(obs) & np.isfinite(sim)
    return obs[m], sim[m]


def nse(obs, sim):
    o, s = _limpiar(obs, sim)
    return 1 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)


def log_nse(obs, sim, eps=0.01):
    o, s = _limpiar(obs, sim)
    o, s = np.log(o + eps), np.log(np.maximum(s, 0) + eps)
    return 1 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)


def kge(obs, sim, componentes=False):
    o, s = _limpiar(obs, sim)
    r = np.corrcoef(o, s)[0, 1]
    beta = s.mean() / o.mean()                       # sesgo de volumen
    gamma = (s.std() / s.mean()) / (o.std() / o.mean())   # sesgo de variabilidad
    valor = 1 - np.sqrt((r - 1) ** 2 + (beta - 1) ** 2 + (gamma - 1) ** 2)
    return (valor, r, beta, gamma) if componentes else valor


def pbias(obs, sim):
    o, s = _limpiar(obs, sim)
    return 100 * np.sum(s - o) / np.sum(o)


def rmse(obs, sim):
    o, s = _limpiar(obs, sim)
    return np.sqrt(np.mean((s - o) ** 2))


def resumen(obs, sim):
    """Devuelve las cinco métricas en un diccionario."""
    return {"NSE": nse(obs, sim), "logNSE": log_nse(obs, sim), "KGE": kge(obs, sim),
            "PBIAS_%": pbias(obs, sim), "RMSE": rmse(obs, sim)}


# Verificación: una simulación perfecta debe dar NSE = KGE = 1 y PBIAS = 0
q = df["Q_mm"].values
print({k: round(v, 3) for k, v in resumen(q, q).items()})

## 4.2 Cuatro modelos, cuatro respuestas

Aquí están cuatro combinaciones de parámetros para la misma cuenca y el mismo modelo. Tres de ellos salieron
de optimizar una FO distinta. Corra la celda y **lea la tabla con cuidado**.

In [ ]:
COMBINACIONES = {
    "A · a priori":      dict(cmax=412.33, b=0.173, alpha=0.813, Ks=0.0400, Kq=0.559),
    "B · optimo NSE":    dict(cmax=139.69, b=0.100, alpha=0.556, Ks=0.0217, Kq=0.535),
    "C · optimo logNSE": dict(cmax=141.81, b=0.100, alpha=0.461, Ks=0.0429, Kq=0.511),
    "D · sesgo cero":    dict(cmax=131.82, b=0.150, alpha=0.776, Ks=0.0674, Kq=0.811),
}

sim = {}
filas = {}
for nombre, par in COMBINACIONES.items():
    s = pd.Series(hymod(df["P_mm"].values, df["ETP_mm"].values, **par), index=df.index)
    sim[nombre] = s
    filas[nombre] = resumen(df.loc[CAL, "Q_mm"], s.loc[CAL])

tabla = pd.DataFrame(filas).T
tabla.round(3)

In [ ]:
# Las tres componentes del KGE, por separado
comp = {n: dict(zip(["KGE", "r (correlacion)", "beta (volumen)", "gamma (variabilidad)"],
                    kge(df.loc[CAL, "Q_mm"], s.loc[CAL], componentes=True)))
        for n, s in sim.items()}
pd.DataFrame(comp).T.round(3)

## 4.3 Las mismas simulaciones, vistas de tres formas

La tabla anterior no dice **dónde** falla cada modelo. Estas tres gráficas sí.

In [ ]:
colores = {"A · a priori": "#bfbfbf", "B · optimo NSE": "#1f4e79",
           "C · optimo logNSE": "#2e7d32", "D · sesgo cero": "#c00000"}

fig, axs = plt.subplots(3, 1, figsize=(12, 11))

# (1) Hidrograma en escala lineal: manda el pico
d = df.loc["2014-04-01":"2014-10-31"]
axs[0].plot(d.index, d["Q_mm"], color="k", lw=1.6, label="observado")
for n, s in sim.items():
    axs[0].plot(d.index, s.loc[d.index], color=colores[n], lw=1.1, label=n)
axs[0].set_title("(1) Escala lineal: se ven los picos"); axs[0].set_ylabel("Q (mm/dia)")
axs[0].legend(fontsize=8, ncol=3)

# (2) Hidrograma en escala logaritmica: manda el estiaje
axs[1].plot(d.index, d["Q_mm"], color="k", lw=1.6, label="observado")
for n, s in sim.items():
    axs[1].plot(d.index, s.loc[d.index], color=colores[n], lw=1.1, label=n)
axs[1].set_yscale("log")
axs[1].set_title("(2) Escala logaritmica: se ve el estiaje"); axs[1].set_ylabel("Q (mm/dia)")

# (3) Curvas de duracion
pr, qq = curva_duracion(df.loc[CAL, "Q_mm"])
axs[2].plot(pr, qq, color="k", lw=1.8, label="observado")
for n, s in sim.items():
    pr_s, q_s = curva_duracion(s.loc[CAL])
    axs[2].plot(pr_s, q_s, color=colores[n], lw=1.1, label=n)
axs[2].set_yscale("log")
axs[2].set_xlabel("Probabilidad de excedencia (%)"); axs[2].set_ylabel("Q (mm/dia)")
axs[2].set_title("(3) Curvas de duracion: todo el regimen en una figura")
axs[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## ✏️ Ejercicio 4 — Elegir la función objetivo

Mire la tabla de la sección 4.2 y las gráficas de la 4.3, y responda:

**4a.** El juego **D** tiene `PBIAS = 0.00 %`: reproduce el volumen total de agua de forma exacta.
Sin embargo su `NSE` es prácticamente cero. Recuerde qué significa un NSE de cero y explique por qué
un `PBIAS` perfecto **no** garantiza un buen modelo.

> ✏️ *(escriba aquí)*

**4b.** Suponga tres propósitos de modelación distintos (caja 1 del protocolo). Para cada uno diga qué
FO usaría como criterio principal y por qué, en una línea:

| Propósito | FO principal | Por qué |
|---|---|---|
| Diseño de un vertedero de creciente | ✏️ | ✏️ |
| Evaluar la oferta hídrica en época seca para una concesión de agua | ✏️ | ✏️ |
| Cerrar el balance hídrico anual de la cuenca | ✏️ | ✏️ |

**4c.** ¿Cuál de los cuatro juegos escogería usted si el propósito fuera **operar un embalse de
abastecimiento**? Justifique en dos líneas usando al menos dos métricas de la tabla.

> ✏️ *(escriba aquí)*

---
# Parte 5 · Estimación de parámetros por ensayo y error

> **Protocolo:** caja *Estimación de parámetros*. Esta es la calibración **manual** de la diapositiva
> correspondiente: el ajuste es subjetivo, depende de la habilidad del modelador y requiere
> normalmente diez o muchas más iteraciones.

La herramienta es una sola función: cambia parámetros, corre el modelo, calcula las métricas **en el
período de calibración**, dibuja y guarda el resultado en una bitácora.

In [ ]:
bitacora = []      # aqui se van acumulando todos los intentos


def probar(cmax, b, alpha, Ks, Kq, nota="", graficar=True):
    """Corre HYMOD, evalua en el periodo de calibracion y registra el intento."""
    par = dict(cmax=cmax, b=b, alpha=alpha, Ks=Ks, Kq=Kq)
    faltan = [k for k, v in par.items() if v is Ellipsis]
    if faltan:
        raise ValueError("Reemplace los '...' por valores numericos en: " + ", ".join(faltan))
    s = pd.Series(hymod(df["P_mm"].values, df["ETP_mm"].values, **par), index=df.index)

    m = resumen(df.loc[CAL, "Q_mm"], s.loc[CAL])
    bitacora.append({"intento": len(bitacora) + 1, **par, **m, "nota": nota})

    if graficar:
        d = df.loc[CAL]
        fig, ax = plt.subplots(figsize=(12, 3.8))
        ax.plot(d.index, d["Q_mm"], color="k", lw=1.2, label="observado")
        ax.plot(d.index, s.loc[CAL], color="#1f4e79", lw=1.0, label="simulado")
        ax.set_ylabel("Q (mm/dia)")
        ax.set_title(f"Intento {len(bitacora)}  |  NSE={m['NSE']:.3f}   logNSE={m['logNSE']:.3f}   "
                     f"KGE={m['KGE']:.3f}   PBIAS={m['PBIAS_%']:+.1f}%   {nota}")
        ax.legend(); plt.tight_layout(); plt.show()

    return s


def ver_bitacora():
    return (pd.DataFrame(bitacora)
              .set_index("intento")
              .round({"cmax": 1, "b": 3, "alpha": 3, "Ks": 4, "Kq": 3,
                      "NSE": 3, "logNSE": 3, "KGE": 3, "PBIAS_%": 1, "RMSE": 4}))


print("Listo. Use probar(...) tantas veces como quiera.")

## 5.1 Tres intentos guiados

Los tres primeros los hacemos juntos, para ver cómo se mueve cada parámetro.

In [ ]:
_ = probar(cmax=412.33, b=0.173, alpha=0.813, Ks=0.040, Kq=0.559,
           nota="punto de partida a priori")

In [ ]:
# El modelo subestima el volumen: hay demasiado almacenamiento de suelo, se evapora de mas.
# Bajamos cmax.
_ = probar(cmax=200.0, b=0.173, alpha=0.813, Ks=0.040, Kq=0.559,
           nota="bajar cmax -> menos ET, mas escorrentia")   # NSE pasa de 0.29 a 0.58

In [ ]:
# Ahora el reparto: alpha muy alto manda casi todo al flujo rapido y el estiaje queda vacio.
_ = probar(cmax=200.0, b=0.173, alpha=0.550, Ks=0.040, Kq=0.559,
           nota="bajar alpha -> mas flujo base")

---
## ✏️ Ejercicio 5 — Su turno

**5a.** Haga **al menos cuatro intentos más** buscando mejorar el NSE. Cambie **un parámetro a la
vez** y anote en `nota=` qué esperaba que pasara. Guía rápida de qué mueve qué:

- Volumen total mal (`PBIAS` grande) → `cmax` y, en segundo lugar, `b`.
- Picos muy bajos o muy altos → `alpha` y `Kq`.
- Recesión demasiado rápida o demasiado lenta → `Ks`.

*Meta sugerida: NSE > 0.60 en el período de calibración. El mejor valor alcanzable con este modelo y
estos datos está alrededor de 0.67. No espere una mejora monótona: es normal que un intento empeore
el resultado — eso también es información.*

In [ ]:
# ✏️ Intento 4
_ = probar(cmax=..., b=..., alpha=..., Ks=..., Kq=..., nota="...")

In [ ]:
# ✏️ Intento 5
_ = probar(cmax=..., b=..., alpha=..., Ks=..., Kq=..., nota="...")

In [ ]:
# ✏️ Intento 6
_ = probar(cmax=..., b=..., alpha=..., Ks=..., Kq=..., nota="...")

In [ ]:
# ✏️ Intento 7
_ = probar(cmax=..., b=..., alpha=..., Ks=..., Kq=..., nota="...")

In [ ]:
ver_bitacora()

**5b.** Tome su mejor juego de parámetros y evalúelo en el período de **aceptabilidad**, que hasta
ahora no ha tocado. Este es el paso *Prueba de aceptabilidad* del protocolo.

In [ ]:
mejor = ver_bitacora().sort_values("NSE", ascending=False).iloc[0]
PAR_MEJOR = {k: float(mejor[k]) for k in ["cmax", "b", "alpha", "Ks", "Kq"]}
print("Mejor juego encontrado:", {k: round(v, 3) for k, v in PAR_MEJOR.items()})

s_final = pd.Series(hymod(df["P_mm"].values, df["ETP_mm"].values, **PAR_MEJOR), index=df.index)

comparacion = pd.DataFrame({
    "Calibracion":   resumen(df.loc[CAL, "Q_mm"], s_final.loc[CAL]),
    "Aceptabilidad": resumen(df.loc[ACE, "Q_mm"], s_final.loc[ACE]),
})
comparacion.round(3)

> ✏️ **Respuesta 5b.** ¿El desempeño cae al pasar al período de aceptabilidad? ¿Cuánto?
> Según el protocolo, ¿qué haría usted si el modelo **no** pasa la prueba de aceptabilidad —volver a
> qué caja del diagrama?
>
> *(escriba aquí)*

## 5.2 Reto (opcional): equifinalidad en 300 corridas

La calibración manual sirve para entender el modelo, pero no para explorar el espacio de parámetros.
La celda siguiente hace lo que en el Taller 2 hará SPOTPY con algoritmos serios: muestrea 300 juegos
de parámetros al azar y grafica cada parámetro contra el NSE que produjo (los llamados *dotty plots*).

Lo que va a ver es **equifinalidad**: juegos de parámetros muy distintos que dan desempeños casi
idénticos.

In [ ]:
rng = np.random.default_rng(42)
N = 300
LIM = {"cmax": (1, 500), "b": (0.1, 2.0), "alpha": (0.1, 0.99),
       "Ks": (0.001, 0.10), "Kq": (0.1, 0.99)}

muestras = {k: rng.uniform(lo, hi, N) for k, (lo, hi) in LIM.items()}
obj = np.empty(N)
for i in range(N):
    par = {k: muestras[k][i] for k in LIM}
    s = hymod(df["P_mm"].values, df["ETP_mm"].values, **par)
    obj[i] = nse(df.loc[CAL, "Q_mm"], pd.Series(s, index=df.index).loc[CAL])

mc = pd.DataFrame(muestras); mc["NSE"] = obj
print(f"Mejor NSE de las {N} corridas al azar: {mc['NSE'].max():.3f}")
print(f"Corridas con NSE > 0.60: {(mc['NSE'] > 0.60).sum()}")

fig, axs = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
for ax, k in zip(axs, LIM):
    ax.scatter(mc[k], mc["NSE"], s=8, alpha=0.5, color="#1f4e79")
    ax.set_xlabel(k); ax.set_ylim(-1, 1)
axs[0].set_ylabel("NSE")
fig.suptitle("Dotty plots: cada punto es una corrida del modelo")
plt.tight_layout(); plt.show()

mc.sort_values("NSE", ascending=False).head(8).round(3)

> ✏️ **Respuesta 5c (reto).** Mire los ocho mejores juegos de la tabla. ¿Los valores de `cmax` de
> esos ocho son parecidos entre sí? ¿Y los de `Ks`? Un parámetro cuyos buenos valores están muy
> dispersos se llama **poco identificable**. ¿Cuál de los cinco parámetros le parece el más
> identificable y cuál el menos? ¿Qué implica eso para interpretarlos físicamente?
>
> *(escriba aquí)*

---
# Parte 6: Dos cuencas colombianas

> **Protocolo:** volvemos a la primera caja. Con datos reales, *establecer el propósito* y *evaluar los
> datos* dejan de ser trámites.

Hasta aquí trabajamos con una cuenca experimental de laboratorio: 1.8 km², datos completos, todo
limpio. Ahora vamos a mirar dos cuencas colombianas reales del conjunto **CAMELS-COL**
(Jiménez et al., 2025), con precipitación satelital CHIRPS, ETP de MSWX y caudales del IDEAM.

| | Cuenca **andina** | Cuenca **Caribe** |
|---|---|---|
| Estación IDEAM | 21167080 | 28037030 |
| Departamento | Tolima | Cesar |
| Área | 446.8 km² | 3 434.1 km² |
| Elevación media de la cuenca | 1 560 m | 1 164 m |
| Elevación de la estación | 415 m | 108 m |
| Régimen de lluvias | **bimodal** (abril y noviembre) | **bimodal** (menos marcado) |
| Período del taller | 2001–2008 | 2001–2008 |

Las dos series traen `P` (CHIRPS), `ETP` (MSWX), temperaturas y `Q` observado, ya convertido a lámina
(`Q_mm`) con el área de cada cuenca.

In [ ]:
CUENCAS = {
    "Andina (Tolima, 21167080)": {"archivo": "cuenca_andina_21167080.csv", "area": 446.82},
    "Caribe (Cesar, 28037030)":  {"archivo": "cuenca_caribe_28037030.csv", "area": 3434.05},
}

col = {}
for nombre, info in CUENCAS.items():
    d = pd.read_csv(DATOS / info["archivo"], parse_dates=["fecha"]).set_index("fecha").asfreq("D")
    col[nombre] = d
    print(f"{nombre}")
    print(f"   registros: {len(d)}   dias sin dato de caudal: {int(d['Q_m3s'].isna().sum())}"
          f"   dias sin forzamiento: {int(d['P_mm'].isna().sum())}")

col[list(CUENCAS)[0]].head()

**Lo primero que hay que resolver son los vacíos.** El caudal observado puede tener huecos (los días
sin dato simplemente no entran en el cálculo de las métricas), pero **el forzamiento no puede tener
huecos**: si `P` o `ETP` traen un `NaN`, el modelo lo arrastra y a partir de ahí toda la simulación se
vuelve `NaN`. Hay que rellenarlo y **dejar constancia de que se rellenó**.

In [ ]:
for nombre, d in col.items():
    huecos = int(d["P_mm"].isna().sum())
    d["P_mm"]   = d["P_mm"].interpolate(limit_direction="both")
    d["ETP_mm"] = d["ETP_mm"].interpolate(limit_direction="both")
    print(f"{nombre}: {huecos} dias de forzamiento rellenados por interpolacion lineal "
          f"(el caudal observado se deja con sus vacios)")

## 6.1 Dos regímenes en una figura

El ciclo anual medio (climatología mensual) es la firma de una cuenca.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)

for ax, (nombre, d) in zip(axs, col.items()):
    P_mes = d["P_mm"].groupby(d.index.month).mean() * 30.4
    Q_mes = d["Q_mm"].groupby(d.index.month).mean()
    meses = np.arange(1, 13)

    ax.bar(meses, P_mes, color="#5b9bd5", alpha=0.75, label="Precipitacion")
    ax.set_ylabel("Precipitacion (mm/mes)")
    ax.set_xticks(meses)
    ax.set_xticklabels(list("EFMAMJJASOND"))

    ax2 = ax.twinx()
    ax2.plot(meses, Q_mes, color="#c00000", lw=2, marker="o", label="Caudal")
    ax2.set_ylabel("Caudal (mm/dia)"); ax2.grid(False)

    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
    ax.set_title(nombre)

plt.tight_layout(); plt.show()

## 6.2 La prueba de consistencia que hicimos en la Parte 2, ahora con datos reales

Misma idea de antes: $ET_{real} \approx P - Q$ debe ser menor que la $ETP$. Con datos de laboratorio
esto se cumple siempre. Con datos reales, no.

In [ ]:
filas = []
for nombre, d in col.items():
    P_, E_, Q_ = d["P_mm"].mean(), d["ETP_mm"].mean(), d["Q_mm"].mean()
    filas.append({
        "cuenca": nombre,
        "P (mm/dia)": P_, "ETP (mm/dia)": E_, "Q obs (mm/dia)": Q_,
        "aridez ETP/P": E_ / P_,
        "C escorrentia": Q_ / P_,
        "ET implicita = P-Q": P_ - Q_,
        "ET implicita / ETP": (P_ - Q_) / E_,
    })
pd.DataFrame(filas).set_index("cuenca").round(2)

**Lea la última columna.** En la cuenca andina la evapotranspiración que exige el balance de masa
es apenas una fracción muy pequeña de la ETP reportada, con un índice de aridez de 1.84 y un
coeficiente de escorrentía de 0.65 al mismo tiempo. Esa combinación es **físicamente difícil de
sostener**: una cuenca donde la atmósfera demanda casi el doble de lo que llueve no debería entregar
dos tercios de su lluvia al río.

La conclusión no es que los datos sirvan o no sirvan, sino que **hay una inconsistencia que el
modelador tiene que declarar antes de calibrar**. Las causas candidatas son varias: CHIRPS subestima la
lluvia orográfica; la ETP de MSWX puede estar sobreestimada en el trópico; o la curva de gastos de la
estación está desactualizada. En la cuenca del Caribe, en cambio, las tres series son mutuamente
coherentes.

Por eso, para la cuenca andina el modelo se calibró con un **factor de corrección de ETP**
($f_{ETP} \approx 0.43$), una práctica estándar cuando se sospecha un sesgo en el forzamiento (y que
hay que reportar siempre, porque es una hipótesis, no un parámetro del modelo).

## 6.3 Correr HYMOD en las dos cuencas

Estos parámetros salieron de una calibración automática sobre 2002–2005 (con 2001 como calentamiento).
El período 2006–2008 no se usó para calibrar.

In [ ]:
PAR_COL = {
    "Andina (Tolima, 21167080)": dict(cmax=55.1,  b=0.841, alpha=0.359, Ks=0.0592, Kq=0.990, f_ETP=0.43),
    "Caribe (Cesar, 28037030)":  dict(cmax=208.5, b=0.353, alpha=0.558, Ks=0.0471, Kq=0.512, f_ETP=1.00),
}

CAL_COL = slice("2002-01-01", "2005-12-31")
ACE_COL = slice("2006-01-01", "2008-12-31")

resultados = {}
for nombre, d in col.items():
    p = dict(PAR_COL[nombre]); f_etp = p.pop("f_ETP")
    d["Q_sim"] = hymod(d["P_mm"].values, d["ETP_mm"].values * f_etp, **p)
    resultados[nombre] = {
        "NSE cal (diario)":  nse(d.loc[CAL_COL, "Q_mm"], d.loc[CAL_COL, "Q_sim"]),
        "NSE ace (diario)":  nse(d.loc[ACE_COL, "Q_mm"], d.loc[ACE_COL, "Q_sim"]),
        "NSE ace (mensual)": nse(d.loc[ACE_COL, "Q_mm"].resample("MS").mean(),
                                 d.loc[ACE_COL, "Q_sim"].resample("MS").mean()),
        "PBIAS ace (%)":     pbias(d.loc[ACE_COL, "Q_mm"], d.loc[ACE_COL, "Q_sim"]),
    }
pd.DataFrame(resultados).T.round(3)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(13, 8))
for ax, (nombre, d) in zip(axs, col.items()):
    v = d.loc[ACE_COL]
    ax.plot(v.index, v["Q_mm"],  color="k", lw=1.0, label="observado")
    ax.plot(v.index, v["Q_sim"], color="#1f4e79", lw=1.0, label="simulado")
    ax.set_ylabel("Q (mm/dia)")
    ax.set_title(f"{nombre} - periodo de aceptabilidad 2006-2008")
    ax.legend()
plt.tight_layout(); plt.show()

---
## ✏️ Ejercicio 6: Cierre

**6a.** En la tabla de la sección 6.3, compare `NSE ace (diario)` con `NSE ace (mensual)` para la
cuenca andina. La diferencia es grande. Relacione ese resultado con la diapositiva del protocolo que
ordena la evaluación así: *(1) volumen anual, (2) volúmenes estacionales, (3) caudales semanales y
diarios, (4) hidrogramas de creciente*, con **énfasis inicial en 1 y 2**. ¿Por qué ese orden y no el
contrario?

> ✏️ *(escriba aquí)*

**6b.** La cuenca del Caribe, que a primera vista parece la más difícil (más seca, más grande, con
vacíos de datos), se modela **mejor** que la andina. Proponga dos explicaciones: una sobre el
**régimen hidrológico** y otra sobre la **calidad del forzamiento**.

> ✏️ *(escriba aquí)*

**6c.** Usted debe entregar un estudio de disponibilidad hídrica para una de estas dos cuencas.
Con lo que vio hoy, escriba tres líneas dirigidas al cliente: qué le puede garantizar del modelo, qué
no, y qué dato conseguiría primero si tuviera presupuesto para una sola campaña de campo.

> ✏️ *(escriba aquí)*

---
# Entrega

Guarde el cuaderno (`Ctrl + S`), **reinicie el kernel y córralo completo de arriba a abajo** para
verificar que no queda ningún error (`Run All`). Después, desde la terminal de VS Code, en la carpeta
del repositorio:

```bash
conda activate hidro
git status                      # revise que solo aparezca lo que quiere subir
git add notebooks/ resultados/
git commit -m "Taller 1 resuelto"
git push
```

Si es la primera vez que sube este repositorio, siga las instrucciones del `README.md`.

**Lista de verificación antes de entregar:**

- [ ] El cuaderno corre completo sin errores, con el kernel reiniciado.
- [ ] Las seis celdas ✏️ de código están completas (Ejercicios 1a, 2 y los cuatro intentos de 5a).
- [ ] Las diez respuestas escritas ✏️ están redactadas (1b, 3a, 3b, 4a, 4b, 4c, 5b, 6a, 6b, 6c).
- [ ] La bitácora de calibración tiene al menos siete intentos con su nota correspondiente.
- [ ] El repositorio **no** incluye la carpeta `datos/` con archivos pesados que no sean los del taller.
- [ ] El repositorio es público, o privado con el profesor agregado como colaborador.

---

### Fuentes de los datos

- **Cuenca de ejemplo (Partes 2 a 5):** conjunto `hymod_input.csv` distribuido con SPOTPY.
  Houska, T., Kraft, P., Chamorro-Chavez, A. & Breuer, L. (2015). *SPOTting Model Parameters Using a
  Ready-Made Python Package.* PLoS ONE 10(12): e0145180.
- **Cuencas colombianas (Parte 6):** Jiménez, D. A. et al. (2025). *CAMELS-COL: A Large-Sample
  Hydrometeorological Dataset for Colombia.* Earth System Science Data (en revisión).
  Datos bajo licencia CC-BY 4.0, disponibles en Zenodo: https://zenodo.org/records/18794895
- **Modelo HYMOD:** Boyle (2001); Wagener et al. (2001). Implementación adaptada de la versión en
  Python incluida en SPOTPY.